# Training a Potential: how to setup data pipeline, model and trainer 

# Training a Nerual Network Potential

This section introduces how to use `MolPot` to train a nnp

## Preparing and loding data

Before we can start training neural networks with `MolPot`, we need to prepare our data. 

In [ ]:
%load_ext autoreload
%autoreload 2

import logging, sys
logging.basicConfig(level=logging.DEBUG)

import molpot as mpot
import torch
from molpot import alias

In [ ]:
# 1. get rMD17 dataset
rmd17_ds = mpot.dataset.rMD17(
    molecule="aspirin",
    save_dir="data",
    device="cpu",
    preprocess=[
        mpot.pipline.NeighborList(cutoff=5.0),
        # mpot.pipline.AtomicDressing(mpot.dataset.rMD17.atomrefs)
    ],
)

In [ ]:
data_inspector = mpot.inspector.DataInspector(rmd17_ds)
data_inspector.summary()

In [ ]:
train_ds, valid_ds = torch.utils.data.random_split(rmd17_ds, [.95, .05])

train_dl = mpot.DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=False,
)
eval_dl = mpot.DataLoader(
    dataset=valid_ds,
    batch_size=2,
    shuffle=False,
)

## Setting up the model

In [ ]:
pinet = mpot.potential.nnp.PiNet(
    depth=5,
    basis_fn=mpot.potential.nnp.radial.GaussianRBF(10, 4.0),
    cutoff_fn=mpot.potential.nnp.cutoff.CosineCutoff(4.0),
    pi_nodes=[64, 64],
    ii_nodes=[64, 64, 64, 64],
    pp_nodes=[64, 64, 64, 64],
    activation=torch.nn.Tanh(),
    rank=1
)
readout = mpot.potential.nnp.readout.Atomwise(
    [64, 1],
    from_key=("pinet", "p1"),
    to_key=("predicts", "energy")
)
model = mpot.potential.PotentialSeq("pinet", pinet, readout)

In [ ]:
from ignite.metrics import MeanAbsoluteError, BatchWise
from pathlib import Path

loss_fn = mpot.MultiTargetLoss(torch.nn.MSELoss(), [("energy", "energy", 1.0)])

trainer = mpot.PotentialTrainer(
    model,
    optimizer=torch.optim.Adam(model.parameters(), lr=1e-4),
    loss_fn=loss_fn,
    device="cuda",
)
trainer.add_evaluator(eval_dl, max_epochs=2, epoch_length=10)

def output_transform(data):
    return (data[0]["energy"], data[1]["energy"])

trainer.add_metric(
    "mae",
    MeanAbsoluteError(output_transform=output_transform),
    target="all",
    usage=BatchWise(),
)

trainer.attach_progressbar()
trainer.attach_tensorboard(log_dir=Path("rmd17_log"))
# basic_profiler = trainer.attach_basic_time_profiler()
# handler_profiler = trainer.attach_handlers_time_profiler()

# trainer.run_tensorboard_profiler(train_dl, log_dir=Path("rmd17_profile"))

# state = trainer.run(train_dl, max_epochs=2, epoch_length=10)
# basic_profiler.print_results(basic_profiler.get_results())
# handler_profiler.print_results(handler_profiler.get_results())
# state

In [ ]:
trainer.reset()
state = trainer.run(train_dl, 10)
state.metrics

In [ ]:
state